# NEXUS-SLM — GPU Fine-Tuning Workflow for Qwen2.5-0.5B-Instruct

**Project:** NEXUS-SLM (Neural Execution for Unified Support Intelligence)  
**Experiment:** Next Justified ML Experiment — Qwen2.5-0.5B LoRA Supervised Fine-Tuning  
**Base Model:** `Qwen/Qwen2.5-0.5B-Instruct` (494M parameters, 151k multilingual vocabulary)  
**Seed:** 42  
**PEFT Configuration:** LoRA rank $r=16$, alpha $\alpha=32$, target modules (all linear projections: attention + MLP)  
**Target Environment:** Kaggle (NVIDIA T4 or P100 GPU), Google Colab (T4), or any CUDA-enabled Linux/Windows environment with $\ge 8\text{ GB}$ VRAM.  

---
### Purpose & Context
The canonical baseline model `nexus-qlora-v1` (SmolLM-135M) is preserved with Accuracy 0.4286 and Macro-F1 0.4017. Due to a 49k vocabulary ceiling, SmolLM experiences severe BPE byte fallback on Devanagari Hindi (14.29% accuracy). Qwen2.5-0.5B breaks this bottleneck with a 151k vocabulary and native Indic subwords.

This notebook provides a 100% reproducible GPU training pipeline that:
1. Executes hardware and GPU preflight checks (aborts cleanly if GPU is missing).
2. Installs verified dependencies.
3. Validates training ($N=316$) and validation ($N=44$) split hashes.
4. Implements **completion-only loss masking** (prompt tokens and padding assigned label `-100`).
5. Runs a forward/backward smoke test before full training.
6. Trains for 3 epochs with mixed precision (fp16/bf16) and gradient accumulation.
7. Exports the LoRA adapter, tokenizer, hashes, and evaluation metadata.
8. Prints instructions to retrieve trained weights back to the local workspace.

## 1. Hardware & GPU Preflight Check
Verifies CUDA availability and VRAM capacity. Halts execution cleanly if no GPU is detected.

In [ ]:
import os
import sys
import platform
import psutil
import torch

print('=' * 70)
print('NEXUS-SLM GPU PREFLIGHT VERIFICATION')
print('=' * 70)
print(f'Platform:       {platform.system()} {platform.release()} ({platform.machine()})')
print(f'Python:         {platform.python_version()}')
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'FATAL: CUDA GPU is required for Qwen2.5-0.5B LoRA training.\n'
        'On Kaggle: Go to Settings -> Accelerator -> select "GPU T4 x2" or "GPU P100".\n'
        'On Colab: Go to Runtime -> Change runtime type -> select "T4 GPU".\n'
        'CPU-only training is blocked to prevent out-of-memory paging and timeouts.'
    )

device_name = torch.cuda.get_device_name(0)
total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
free_vram_gb = round(torch.cuda.mem_get_info()[0] / (1024**3), 2)
bf16_ok = torch.cuda.is_bf16_supported()

print(f'Active GPU:     {device_name}')
print(f'Total VRAM:     {total_vram_gb} GB')
print(f'Free VRAM:      {free_vram_gb} GB')
print(f'bfloat16:       {bf16_ok}')
print('=' * 70)
print('GPU Preflight PASSED!')

## 2. Dependency Installation
Installs the required Hugging Face ecosystem libraries.

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers==4.46.2 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 scikit-learn==1.5.2 psutil safetensors

## 3. Dataset Setup & Integrity Verification
Loads the verified splits and checks SHA-256 hashes against canonical records:
- `train.jsonl` ($N=316$): `11df262072d7243f623bfd829b407a8dbd2c5a3087ed697a836a7152c7688967`
- `validation.jsonl` ($N=44$): `2e349ad3b1330aec224c266460d2e4189b94138a1d63146a10bd8f45ea65508e`
- `test.jsonl` ($N=42$): `edf3fa5cd949a149ad5d759d81553d9704467491877d0f3c8b7cc5a887a9eda4` (Locked, historical evidence only)

In [ ]:
import hashlib
import json
import gzip
import base64
from pathlib import Path

# Determine workspace root
WORK_DIR = Path('/kaggle/working/nexus_training') if Path('/kaggle/working').exists() else Path('./nexus_training')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'train').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'validation').mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'test').mkdir(parents=True, exist_ok=True)

def compute_sha256(p: Path) -> str:
    if not p.exists(): return 'NOT_FOUND'
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(8192): h.update(chunk)
    return h.hexdigest()

train_file = DATA_DIR / 'train/train.jsonl'
val_file = DATA_DIR / 'validation/validation.jsonl'
test_file = DATA_DIR / 'test/test.jsonl'

# 1. If running in cloned repository, copy from local files
repo_train = Path('data/train/train.jsonl')
repo_val = Path('data/validation/validation.jsonl')
repo_test = Path('data/test/test.jsonl')

if repo_train.exists():
    import shutil
    shutil.copy(repo_train, train_file)
    shutil.copy(repo_val, val_file)
    shutil.copy(repo_test, test_file)
    print('Using local dataset files from cloned repository.')
else:
    print('Standalone cloud runtime detected. Self-extracting canonical verified datasets...')
    EMBEDDED_TRAIN = 'H4sIAE27y2oC/+19224bV7rm/QDzDsvZA2RfKJoqnpl9QbAlxtbYlgVJ7rSvjBJZEisiWeyqohVmYwB3NuIDshvuQY/aPdme8QQtuw1HcQwBSdvARvIqxn6B6UeY/1+HqlXFWnLRpKqKUS4Si2fW4lr/8fu/75/fszrvffieZ7X3Te+mpml6sfre0nue+akHd690zfa+PfLI0NgziT00B2TfcDyTdC1iDIdk13baJmn3bBfussknhmeQrmEtwzv0jMHeCF4F79K1Bns9y+3CvW3DM/dsZ4wfaba7A6tt9G5arjsy4cGhY9mO5eGDfbNjjfpwn2P+dmQ5pnuzO+obg/c+3DV6rrn0nmuP4JPhie544HVN+Prw3FtGz+rAB3T8p7UdE2/fNPBiClqh8oGufaBVtnX9w0LhQ726XC3XijX9vf/+n//TP0cXol6sBwuxRtrGYGB7xGi3Tdcl/TEZOvau1TPJrmP3iUEG5gHpmLestrlEbpmOtQuX5ln2gJh9w+rBw3AnMRzHumV2lskaLNMtk+yYsKJte+AZbQ8WibRHrmf34YnuaDi0HQ8XmMD1kZ7heqRIOsbYJQeW1yUDm8Cq2L0RfkZkvc245YYvbo8G3k12ARmtdqmoV2NXW9c1LVjtdVjLoeEeOB2yYwzgPYln9U3imu0RfmPShv/DEhukb3htWAvYjbD7HKNrJNp58Nbuge10bsKVml5WS1HRq+XYjVcr1yeWgn5feTEas15o19rrxlym54zSucpqpRBc5YaBd5NhlxqQkQF73iDGCA7MoGM4YHOIa8DXI3vG2NjBB5fJZccajg1iufs2GTl75sDrjeEA7ndNa0DgCBnwql2LDEb7LryQb4/Exqlj9uCgOuNMbVOlUCrE26ZKXTotb44O3zz5/Zujl2+e3H1z9PWbo5/eHD0meO/Rj/ReeOxruP0KHod/nr558iX8c8wehb+ev3lyz3/0kL324ZujB0v05pN/wWccfQdPh9tHb578Dt/z6HP4G24/ou8CN75ir2Xf4fM3T44SvPgV/esR/YL/h77RM7j3hL0F/PUF3EEfhNc/YW/xgr7F/2bf9T5982/hesQSHLFnfUPvgg96wV7C3+8HujRsWb5jX4F+ZfyU23QhcPX+nT7/IdwFVzGxVzpW9IiN+7D3bu6CkR85ZiZnTNMrtVijqhVLkW0S+j3YyrxkN+7zR3DlHorfMljSP9FV4zeO6Q/3NPSTw43/yfYT/Kyv6X2P2M+EeylmO76GnUbvenCuNozwwxg0ZbRhKqVqqRBrV7S6HmyYK/A2EHVYg1s2RDVgU8loYH46NNvw7qTdNcDousTodMBgm2hYB0sEHu7BH2Tf+sQam2dtpHesXg8eSGid57yI5XKtXo/3bFpdPnXilNEd+Fzs4Ntitx+xHfelsMjw3BOCOwyfCHe9bCTeUnGhXUomqKRX4hejWJE81TZEshCOQHx7YLgYUbfNXg82086YBrmwIWBbDbwlMuyZBqQUEGKLaNsxd0eDDrH6+PvCt+mNswii2be4iasJRyOTldb1ajU+bCxI264LC2sT13KI2uzTDfk/fJPMjPchs2zfS+b/hW9pH4s44x41hD/RPfqQmWBmUu9So/mM+YijkHn/++M/f/3WrUx3x03PgXhw4mSns8BlTa/Fb+VCsRws8I5js6W8Rxfhmb+asEZffkCdxE+BN3ry+YTXpJEYPof/Kk+EJ8L78NFv2S8R+rUUIZ0qgku87O5ox4WkaojnIJNFL1QK8bsalj1Y9Osba4QHXHC4qbXbHfVIx9rvGgkTolOiNcwqsR6R4SHWdV3aY8PeZwQO8pBs8FLD0HTg6wgviubuFuaD+4YzwP8Tm/6AZGBAQty3esybQuaHZhGsKDjkCxeSLNOpMUrPPkghRClXwXHEn8LiPL0rexX+80d6dm/T03SbWcIv2Sl7RgPa2ywmvEPo4X7NjutrPHlTRH0ZumhV6UUrlaMres8Pp5/Tq/+RrdKPNAL/XjZAYKQ+Z0v60Ldkr+l9v4tLH1/SZXxFf7PH1G9wJ3IYtXT3xO2T4Bf61s8qmEtpNC40LiTI0N6h2pNaGaSsSSnaVYyIhYHbNzxa75jdsGUdmVS1olxJdUcuJhFk1zFGnVEPr5WaMviyjQQxGH0Z/JgY2mVlwJVVY10v6fJh+o6ehed029+BXR3y2Xf9XNMv2DwOnSS+7/8Af4Xy0pjXfsMOi38wWWzxJ3HsIvbql+KM8lTMvc6rPBaaXAFdMZwOuGrSscmOYThgBSy4tU9NgLnEsyZjBxw7/ocVbywGLy+8bSiVSnLWAomgafSJSC9esd+eR9chByEKUtIj/rY8+ivfTCIEeMmOYXD6YB/fpyftJ7av2VY7ZA+yHfosGicc01N1Eg3cxZH6++P/9eCtOzIPxqtSrVaL8TtS7jxcHRMetWCrzzQcl/RseGanMWO3Ket4RytpygjyWCR0h9EthhunsRDlYOVZK5alC/8o8L7MuAx7RtukzZeuYWEtxiRtbpQ+sUnfsCDHoJnFvjVOZnhy4anrJa2gSLUqocgL0qsR1p9cmkpd3Nr2K58d+2DQs40Ou3xRpcSrMnqwPJmUOVPKy8vFWnyluFisyYdoIkR5JbKEr2RHfjcIal6xQgd9zjfsNothnrLD9nuprXWbWWMe04iAhroGtL4804DY49/8WtRE3vaatb14KHR+YqBTYRap5PLlaq2s8DdaNa5kuUYGptkhns1LxNxA/UO9VtBLxHDhCa7ZY30IrOseOPZgj4iWLbYk4GLcZSx9fZvAV7GaI/uoTAx2Ta/W4tenVg1XHDccuzOC1GnfIgeG4xgDBEP0DKsf1ID2wRCJunnXZnicRsxSxNsasF1grno3fzuClcyi7qOMTYpyIXBG53XGBjvXfg8cnxTkXaOrx48Z7iG2X5JcZebHpq7XSvElwnoppohK47mX1LYesxyaOZcvqVFl5vkxNcdohE9CVflHIhu5G1P0B3/wr6yXHUoNlojoAnxDn/01r98T8WHc6TBf95jVxOAmd2oN8uE/vtW4n9aPSg/JVNArqmq2FHC2Ph3Ct+igcaI2DFuAbteC1IKa+r75T3BOB6Ndo+2NHOzj4feAI0q8A5uMaf5h9zKBzvFvfLNt92l7O5PGn7oiXqpK251ZM1gkahHRP8IKMhu5MyZ9y/WMfdNvsPKDb3lyYxWWFtYOwl7LHSK+brJrMKUbTW8nVpU17kJcjds/wg/pgT9iURY/5nA8XzcSNizzGDwUq3pcmisV81nw/YrVNA55YHz0O98AndLwDRVGggD2Kx9U9lT0P78U+QBvTuId9/zmwtd+KyAarN9LXOA/rWccH7TMHUSs7Blrcpp0nZ7NDwlNNkVtxUVYjTHyuvCVP8MDC1GdizbNHoh4xj+u4jW7jml+Rvt+9oXzHcsUpBy+OfLsDzrmDlgze3eXRnoUR7oLKQUWTeUmNzH6dCVFkyXRWqi75OnB1ZWN8rLceJiv+U/f42ZuVZWOpFiRnO2qSDlhWb2RK1JQWHcKSfBPGhkaDumPPoG1GYKVwlkK0aM3LmSIZE4pclHCmPH+AAPO9iS9YGHo6GRJY0EgC0pUZbGovR3Qr55ukIqOZ5255qJXrjx7BRlYybcLmz3qj8nINZ1pOhM53CiQmdfCeO1F64Gdj5JqvmOiQkGLq+/0zKC+w33R/ng02Lcg/ulBFMSrpryyipZjj8VPywtRDFKlQXqohhrUmK/SFTCGsCB9CGm6PRzoM/tg4cJxYrQ0thwPqJwqWkwpKSlWivGl1KoM+AojhRfN4Fy4sOCnVSvVYmqVK/DBFhxckdSCP+/vwG1rQD89El4uEQzxRBCKDQBe848BJCjPbrb5c0WrxG/VajEWHhKPRjwVXvhAOIxX4uXf+VvsKA4ntcT2It9xofeKIhHj+ysLhUTU9Wohtmr+TIajxSx6sNyhNu+hjKJlXviHAPoeWunPWdGJBQGHvnl4QJ/2UoW5TXL28zwAW9FD8+UHBuQ7nh3BePdMj0a3PhJn0CEQ5GMtqNejU+im49oDo4fVciPLufI8Jl6Qrcvx9PdS1fNvzFn5jaATPxYNubXnNCoMhmvZjg4NW/D5x6XYSZvgAx9SD8b+/kHyf9OBmd+hBDVvVwbZmaYYHizLG9ox2yaSHRADtlcfriho/eAutgbgfPGz4HHH9EbOYInsjDy2AXfhCV3Dhb+Mdheewk7AwJt1Ris1K6sGfWrSlry4YYwp6JMWLTHkNsB54yq0Q8QbbTjxXf9xjMnjd8u04M/slwMnl4NyWmSbYHkMr3YpUQXotN89nTBG2QjVQlgzigR/7ge+wr6gqXjNTNIr0Yv5zg9NDgXY+yl7utxMDozMd+zFL9hfoTYzbekkMzJTg0DmTnxQqZYViJiCFKRsmnuW69F6a9/ewQEpHiYPTXJte4OipA0WJkOIbO/hYDKDVRvOaKqq4lSkKanN1lZqUqb/lpShseiZQEXO48MIsR6dKk4CELslwcMWvaWt6bUQWFuUMRAuCr96uzfqmK4/p48Bn2s6yE/kUoAdpSPiE4SdJIuR7bS9sphRqdRj0paPDHAeY+yCDShxwXDPMTrSpKRAyPnJ8n989X//36sHi9IFVK2GrpUqp8JyDwXwgcFvT1j6+xji0TOAOsz9LCixDrpcJpByV8UsY4jMhOGvnnB/S5PWicqVz1XC3+Ar4a2Pokks50D5Xrhev0b9E00BsC72tYB6/eijzZK8Oaa8il06FcdI1jlZQU8wbfpClBpCSRqr77wQhcAXLNo5FlUdgRfHh5/LidZLETY9laoUb1nrBZ6Zr4SRP1+IGPG+NGlIwfCvhWn4IQgy/VLXAyLIpE7osqsqPOKnCS3/dzFARwEqOvcsP+kRiCk7jVpZiqPHOHC4SrkLyX7X7jF+xx3M0nnOuQTB0y6Lp4YW5PVkp2t+YrB0lDZFkvEKvAVImVYwoQaZFyp6DLpjxx5DwAShZo82gvZNHIRj5I6QaRjGAJv3CF7F5rtDOilAzN+Jny41XEehXIwtbRqjCTSHmcvWmbrGJbOjNnlR1jP7Q9sxYKd0LNfY6ZlR+M+OhVvDcunUxgCTs31zedb8M2tmp5psQiYhta+oPQ24g1jx9TkDuKsIcJairHQxAPpj6jImBzQDg36H9g4g2vrxvLibfM8BaJVCOQkPYcqEVOdja+QZHV2Wi+EKdHRjUVDKlVq1Gu809JLkNFZtMrZHWIC4ZXVMAkGQ6QxoMIHDxDgPg/4BvOUKxBMdg/BG3fXLjQyae6dUglOLMtS14JoWSXMwzWb2/3s8TCIF//BUGAtP8eMxLOysP6P54112/F/TE35HAglEzUbIg0Up2455uxGhMIehJuCiQ1lK5SjvZWAJleCLaJjAkRUJmBbeMZeYe451ylxWZRKu6sD38iRCSzjoHmXFtPYstACC/alvel0bu6SuZxodYu8SOORgGA8gCTG9X7gvI+ms3BaaLSw/48QtzxF9oaadXppTYgBn46hNL0JVXrpekJsp9sAc0/lU2pg3O1TjIUnimmc2cK1QjLAJXGTMkMQgZU0TkUiPtonAgpiOA7bjoIt9VcPDw+TxyGQ0pGwoYMQu31ghHbs9wot2lycmls356X6kxkmhyzS6l+wDZORaEw2k2YoX6fWHlPWLaojhKY5uEOt++1jPQhgKmxwD/7JrOX0+iTJKYQol15gVrTw5b719YH+Ak+twYnCWE+4S8i9tu0On2Kl4DMq/4CGC+2kIEI9fWI7t8ywcz5ZelrL+1m82WivbrVWy2rqy9uvW5g3ycXOL3Ghtbbc2V5s3lsivrm+T7UstsnLt+uZaa5NsbzZXLq+tXyRb283t61vkUnNr/f1tcvXar1urywhkuTDFVHo+099atR4DblbOwbHisk11UVwDkXWU4qRjenBOXAEWxaHg5Sna2e/WqUivXVgKSVjEVIei/WxO9BTio7239DYe9BCFdjShC9Uuo9jvRS/OabXynCdafwYCO4VqWVYnYi4S/RHiVT2aUe3Bmx8Y4yUyNmlCR/3mjtFDfAQHr0KCmo3YV67jUK1WTEbpHt5tqDfXN0P8COIHMcbZxiRZEztGdH6mQP1EiTciFHtR+xk08++wz7jLbGKMWWZke7TyTTg6QySNr9nXWWRylWpYHDDMV9dYEGihmneuVJ2GS+KzhEQSC83FV69VwkI2Ad8gzQD6CAxIxI2bC9enLuMWC+Vw816ASDvooBY+Ay7oISYoGeHEeoy3WZH+PsPQPxO8wGgaae8Rn/nIH267HWD+/hySk4gWtnkDIdRzeOHj0CYa4okngnLA76GVdIUdqRfDZwa9vezAqcc3O8nOTc6DmlpiLH6y8d2zjWdOy4tTK8Ipwf6FsszNTNk7Lc/s80ExzHHPaCRo3vtFPRFUqSagvpkm5U+nb5FtmUDdawurAsXw233jQ66DBm6I6pNbceYQWJf22Ae68g7x52yIMxi+ukNbxj/G2PuHost5zLFN5wdykgtwotIp6SVZUWxAG4MeTbBZHi9mxpdoLg6xTB/C2p2R54G/ggB4NIBnDu2BC5eYTWqfAxKBsnIoSSYRuLa52tokq63t5tqVLbLRvNgiGy2y0lxfaV0hl5vk2sb22rV1st68tEaaTbLZvNRcIhtXWs0t/1lr2/G13mlZaFJbGSUpcVWGS/pltg7Tb+sho7eHoGs68ssKalTtBXIsgnPAuIH8jCOtRsw7BQkpjgSWa7MKlzXOhrRj7qm68sDp8qyH30MQWLrrKmZRe5f8l1JZI3a7PXKwvMs6UtzLL5NfodQL3kPjjYF9cCFZc1eNUEppKEYlzKUV5YArnvY8IS2WmMFP5fDlctyuJFOMXR37BVwGmUBYFsdM+HVxvrEYxQPcycpIPOVxPRxZdbv2AbrXoWF1FqTGXawrjqWul6Vj+XEXIa2Wy6ILatwbjRl7l9k7tKKmxbGIhY6WdFx80ll6MDj67ROj17F8KhCeyMTw3ucTjqZM7qpy90gmWMtxtSkZhVWeq016obQA0ouqcHahlMR0eU5uBZybDGmn1ax2tASGAaxv+P+hBHal0MiMWTpLPIjaomqF+pwUyRLTA2WpLVYtKg5yoSCFatuXWpstsrZFmuvk+vrl9Wsfr5Ot65srl5qbkE9C+nj1BrnS3G7B6qyt//ra2kpr2c8hrzQ31z66MQVaaDqOi/QYO4vRQrrPIM3KFvCl92Fz0DNi9sFiUACCyRRXlgWkyDUHyLY1MA8g3M9d8SIlm6YuXdRKCRp8gQ2DBX5/VcBj3meRrSBWCbjPBmMPT96FGfuCKYX8p7RyqtV3EVCJxQaoZz+j1dG7SkjWJKWDz8V5nub64tLDlNyYOjuslfM/GDwTai9rOjnwE/IKH/uonuBS/+DzODN+VrbKf/TxQBOYx0dCCe3f/RP9IMDq8CfEq59xHc7bITBR4gZ2xj63VCkopmFK+oJYvBlsRR7ARrI01yarC8izbgj6o2FuB75f2wvVk3xA4C7WHLrwrPEUwW+eSwhaNQI3ovVHxx5YbQhMKFULLBIVz4OAimZYGJJhDHhgQ0YwQrm8XTIa7NifwoM5lM3LwYSmJqP4tmAZICwmbJTCJ52l2wwXgLKxmk4fojMqIAXbFHbcYI89EyNwZMfp4lvwMY6Z6uUpTvQqa+aajJUSaT5kFfhrMmJFX4TWhsgVm1ZDu2e1x9hOELlrB7bSELaq25ht4Dxr6tG6TFWrFOJtzKiymzkHkq5VyosQXJyTMD9b1k11cCRzfW+L/gZYiNFgf2AfDBrvWmxJqfakrLSUIqd8F8fywpz8gjWVsYk2fgZDVmW9PCfas8SxV87nf3RZQprPyQrNtq5t96g4Une0JNffWOFtyOtsbEk+wYD0zJTX0iMb1sPln5zi6xa7fSUrAW00t7Y+vra5SjZbW61tcmVt/TLFTl1ZW7lMLjc31ymUit7d+s3G2mZrFQFVrc3Na5sCSkUuNdemAU/lYwcq5VJq8tTYRaPnWWh9WCmWDqhIXJV84ITQYNxF+93tmRLcAyG1rBu9fEawshTTZz0Rtxq9kWB61q9VSLNjNEKiQs6MZY0oqyH+OcUJWUqvi6bgmc8pKxU/Fr1uoWtlLVaaKrcBc6we2ILEm8Vwb38OSj6NxdbakakDZofcnC2SLc91tmJRnyDz4PGew7xFl8/jbAskA0et9U1SAlczQAezb1tkNKRgXtaRR07k5BwM+R7GAVMXLNEVKqPC+YC4nsLQ/BmIqRS1YgIw8aIqByy2ZECpHiF6fCZAcz6pYED9yt3pU+Yg/8xgWupEg7tK3rq5G8gQ4hv7/LL36WdyP3OfyzdhWPUXtimYfNPxBP3INICv08vlKaUnqlq5HqqVR0RjT5TQxtdM1ZWtO/PaPER5wv6Jij5+QQtut4NWLhOQvMf++lzSF3gZ5S64MyWWMd9Wt65HtMCDuPGeH00+FUaIMRt/z6hpfKPFWJC/9uuQ2BB/e9xzKkFbim0JJUWbJvtsSchZdGtiJDQDjD+OUmJFC/UyuZSmaBzOXMlL6YwqDWVdBhDOHcKUQQsx35OEtUolRnVo9mg8qXBzrvvXMl2GlB3nVCYgobBSnkFBhVKE3lNBqE5GqqGw7ojzEfCNKl6z65jmZ2xzXkio9JJz0XW9PlnNnoOYeGMxJMCVENhiuRrTB2K+A2e7nFEyCtR8l9k1+RpnET06Y8ax0/OBlOAQSuxMWa7IxcstNhZeTlEPM2IvCrHauYdAZz4hWymGNk6oQuBrGbIKP680RNulfNEf0fv8Cb6Jav+/+vWgIEpaio+IJgT5XokaBysqvTpnGn15F4rXZ9JMasxU/0gJhqOkRob8YUFY2n7h9Mk8pCvVo7NJp0v9MFP8A+tkB5O/97j9C3ZH9FzdobW3r9kqPqLb4qWfH4TcPcPA3Ds/+yPX48oluYPVYlORjFCOb+vEjHI5L9KHWOXOggQm/RLgO1US5p4hKCHyBb36tuYoJJUO/c812eiAZ/VNnCZgCitGwAR5nsWH4JRGFIlPs+SBMXslmnk/CYDHXwmCBtHysZf8VZiz24FhlrIv3vm7E7H10Vg7JpsLo5mS1hBzTJEkZ7uSbO9kO0UajcGCCWujYFcFGW5Mx6VihthhWZ5CwyTD1nOxpMI+lIohz8HKQX5bBYtiYxM1XDqoBID9E7R6E+QXHDLSNdzB+x7p27eykQXICZ+bUg9F9l6xrK8UZI7NFFwHa8CAnUuhjsq+sYOCAOae0ZjLIGBqmaBaPVfW7eIQpXxP4Ce0hLnG1eqV8vwpN2J25AKSfOrFyGwqb3YGJKl0SNLXuWLNZkFaPITr9ltMvnDAQogDqMcjCvOTi1l0JvS6LOpM46YHfioaRuUyEdyHDPuzKBKSRSXlX6ESKdhNFPW/5bBlnvUnAePkeCBWl7UEL4NfNoyhQs964adi4WK100fkGrNOvaVUO1COvNWr5TlD7P0G2SsBvxRBQrwcW/CBDxk8k/79A/uk88jxo+6NptTgUg8glGt12eNNhkYJQeGnRTopIWqUXr0oD1lIOMM+JGtdbHc7Zh/MEpF/pglS4eQjsTmYN1GPRVdDiJlg3CseO/M1R/qK5OQwka/Lt4PXdG1SO/cdZ0T4iMjiz4bUtUkt3NlANX9//OevfwbwGE0LHZgT6gef0277fRYM8rak8LN8TEHqVLIWvvC4T2gmHmaefU7f7ERABG4HIw7PRG7/Itr0Tyy+l+PIsyAPhMzOS5NBJS77xVVGumW5ChcvEd84M5n3uQcw6iGCamkmlsLGYuBL1ABtGb+3hmELVSinrpeW8Tn3CaXxZxUsxju2ROBIWbtC35zVvtjMANU3T1bCyLNweaUoSbV8hO2aUQ8dW4hsYGQ0fgZAaPlKg6GuVTtE6B2f02NrCGyv0TEIN4vXLzeSqYTk2bVUStPx/yA1t81Cf6GAzZqyLjOdGARmKs6W9VCSLkMhZ9TbWnCK6apemCycw3KcUjifrJt/Jl41a9E8NR+kptAIhRoqTpe3ULr0zIDS5bOF4XNRy6rUq5PK0rBVLm5tNxaYwb5UCF2WO3Kxl052A+9KzSp8X773efyxTJqD8MQS2x7YeWe61MuLAkpKDcdRktUCtkwD8h28vlHPcyFEM2ljarIoP30Un4dZ4LJemn8vMjNoRh6Fz2GNZ6J+ayw0trimy73uA/uDXdgLiNoDgwRWS2RBbRvj5EQUt3nOfMryZL0KKEtBOEtI/DjCxRg5eDRoXbNAXAMyhOHIMQbWvAbdMsbUalVZ5+ySfYCRxpoYcWPpMZqAHcO12mwCDqJbvAcy6hGshonZE1wPWBjPMp0sqk3qzkI66dZpbaQoWOC701Q7g2lcv7d2NyilMqoQ1oj7IcA5hQbuONlOlHbkAX0axfjHiXadGxB/5s04Ze5UkfFYa8RzLGwhGGM8OrcsLFAxaQBKdo+BgNFn1O1Uk3LHwo6Dd0DrVwjThDOUh0Ay10ieQklbhMOZTGYkB3tbzW1aKYQXWizksahKH8oMV4KVCn8EZnPuCZvBwRKvaFB2Elqt6Kqy6c9DAdi4ze7lk0/hJQ/IrBZ8Dki2IerECB04VdfDviXK0rIOyE4PIXsHXTAlBNwpGIksLMa79VXS0+6T0X+rzavNi61VsrF5bfX6yja5unalSbZR5X2ztX19c51srK1cvr5BLl0jF5s3mmSjuQn/rW21yMfNjeaWUIa/0ZqGx/id2i4pajfJvUshI9EZDXuYSpj/ddfYN8nA2u8xKpbGz6AVXixIs7xNY7hvBeejjccOqYFmRQhm3TXVw/SR8QPLD6ixDvr31EO+P/ns9yNwuEMFaD9q0kMBZUCcF+1lhkaZgkT9wi+jzDkAmpbkmng8r0rQeIKTT/u1+2MOQDrbPlMeuFoKlYJqTKb0LrjzaLj63KdhvSuxaPwkGBh9NKpgVnwuBWPs76OYYfAv4wLXmQBxmc9rygX9d5eAnnFqLaVVUHZr9HrlNLNPhHXm5jTQF79PbevvRS71lO2kz31XccT2C9tKhz5rKoeGhfhkHlID+xf6iUeyTT2cbrflX6+tLIPqYmGG84iXUvMBp0xWyH3RDWOMHU9axACTnRBWPX0aNvcY+JQ8TIsQOjMPf9+PjDjP1Wthtn/w6YumwBHnVG1Jl5l95yDt+POQYqwXpp6eYow/jPH8pc/rzSwnM49/C7N432ZsQfHB+aE/Ovs7evdPvvQQIwWKcKwtMn62LKP5f2Xh3D7swk/BxYKhcXAHJjIyWQq4lWv1+LikLHtkPoUL79exKNjIMXcHHdGgga+8ZyGkjVtLWAWva3eINXAhJe3gIZOFaJMctBxQ8iqtbqVWT8J8qcboj42fSU9P14s1uZkQcF2TjtE3UNKVXwAtAQp6ZnyccRH4PQROOsDmd412NyQbm0V5MB/VL+UeLMi8blea262tbbK2/utraystcrVFrq+3frPRWtlurZKVS83Ni60t0lxdJZeut1DNbH0Jxc6uwB/k8tp/W5uuLJhLY6XJDOLXJqApSB7kz5OFUKacNCgOu3K2JQA1TiU9NJAaq1KeEE0IplGZXAoRsWa4bxUIhrCI4mVUkCugiXuwlKBO9kg0Z76aWqw91y3BkqwIjSTj82C7j9HqWLy6fmEigJ2Or5NI3dQjuVrrR6/3Im3AE1FFDcpSx0IJ5EHCzZZnDtBi7ISOpMh0Gi1YtMYXKsgd+hygP8a1oJdIbE56LCbx7rOKPKd0jnmTcwQZyYWol3pkNwq479s7ONBhDFm/Df3pvmkOXUgRDLdLU+2uyYaZ1sDhmmycEOK69j4y9vG276xt2pROlrJJW9LjhEv4SDNLhfYNj9lyylPj0hkFXrmFQHbXcvpc6W5kTKN0l2t4gC6Xpd7amiW/HcGne8jk04W1wRTKRHTkL9IAerkQo/0iZhlsES3wPkC9VtBLKEW0Btusx9JOOsPr2Igr9VUoOh24GHd5QoBkSjh11qQSukwYuoOKVBtSoCRYFbF06c+1oGSVa2B+SolYRYYQSFtNoRmU89popVqL+PzA3flun/F4coaJoLnyJ1Hjexr0Ph7ROx5QX8un6x9LcKlXjOHuGe333fdVF3wdxZ/wtefGl2eZIiuVfwu6rtwR8So8MYiHb31hvKfx7TjO+y3UFkREfiRSxdtMu+OZWPUTX9z60OeVPU9A4XxD9jXZka9GKnp9q8czVM4rOoTXjYZ+8joEozs0EK5/YAzBLfF6y9jMUiY5e76AUiUEcPpKQI/u0B35WAko/RPdlSdy/+Q2259fimLLXXaeuLoOL8B8IJdiDqUc+IdTEBXn5PjlIrlQTeQVyuVIgLN4uXsCud7TEt8US6FqPeti+S1k3o1ZubnTYhYo6IpgsV6fi9pBvNbBwo8BF2WVw0nihcai8BArt7heK0xOKwo3j3M9lKqE5Zx985/iJhYtl4IqxqaBoIpeZw5gisyDn4KWpBAtUGj0EfRcD0WjInBpIbX7Yya0EOpnBIi0V9R43mUAZ94oiem1vBaK9bxAnVQcPc/y9JVSeW6RUWOxfb9eLCZTYjpVv6Mxg4ZGLgyvHAFd31jzi6ruiAYLu6Me6Vj73UD4hSYdxnCIOBzZZwkKcchLlmcur2avaVeUHXZAN/UWIxXTLYuKbvrDEZIw5++FYePjc2rNBL9gwGsELLF/RiL1JaHzEltqm1mbMrUeebVWi2W1FaVHdYSY54qi0h0W5LzV73Ws8Zlf4nUhBGCV6J0xcoV4xn4MJQix+ngp8C0g498xdxEVJjiJ4lrYZl55h5R16Vq1mjhsyNuJ/EXuNmu5W620uHK3SWPQfNfnyro2H8XUM+HrmXvzUa3vrkVkOKg5il+O12w4jC3F97yjNKt0bnqk3CrgTKVQlLn58W4y7Bqegd16LH0vulaFHqJi9536x13TYSk9grsZWnJCv85n70RkbyBnx+nmGjH843kVrVPWxYr1ELrZ6COBHq/+9Ow9qlplS7WvNchMEHFMbzJqvkGHMu2RHSyLzc7omwP2fq1USKpW448x7huORzkoOyPXoQONA+SdQUrXM1O0SW9kvSqT//qHiMdC3FX47dUAdngYgsv+WRDX3w0e+kKk9C+Dfg+XorsbDK2HJnPu+71VP9hi8dJfotOOvPUTUJLEA0gXcLixXn7LUA5pdw1nD6cEaKNSaCaQT7Bz2TO8NKBH0/GLpoVoA38QqsEd+3lIEJL/wddIYERFzN//MdEsV7ZXrcRG6HLMh44eZwY4CTuYeGdAnVzX6O1Sl4hEfi5ON6Fto5x19qfkAH0m58hMYuazDwvUdOZ6sa4sP8YAhuJ2ySHN+v6NWTE5XzyiWR0+eOJzcjyit3ki59NfPgimBONGBV6Jse+/+ZnQdAMCGVP8KndjqRqnUzo1X2jQU2BBO2vdHqs9zKE/JfpXscJh9M9DkYYfh6p4CbxGnplJC7IiXQRD+A4IwplLfZkr7oRohkae/QGbmLd3d8k+lw1uzMZykhJbppripKqdFYl2UmXanDNfazUVeTrxjE/B992yMZDiTCQ494kVb64IRO/eGYEnRE6dXcoFt/yunOsp+UIl4XqxVD2NL7NxRuSUc0/olKNolWrl9PGgyRY2c71PuDtIUujJOVy6punxDbXpKDX8uvqDSKlMRebpkx6JnrYUVU2ojTXiNd0WjbUjzFTMwYqhPjYLcr5SMEqxJJo95xvB0YmvfqpECHCUmoCo+UDjpVCgeuJzUZ3COpWYJzX3ul0lPRbXZoyodIho2YF1P02mczlXLNwp+QolC7cWRrC8i7qnD9x9Gm0WxpiiAIcZruZI2/sF3f1Pp06OcjGPqESsVWWnLJXcFM4rfvm/pUjXk+iyCfRrNN9k5iQgZz4RxbzD6FjyPXH7JLBa3/qgLq68nMiWvxNuMD2+5WIYpjQfiFxjEUBsymJwpVael7itwdVtU2GimLIHN3c/pVTG1QuxodmKGCWm/B50wph3GSwKAAN3g4VemrzDinxCu3bT6OKe6r9TSmCV+VlNZpQXiO2rWIHkvScznKbKWDjsXdK5W7Sb7Om7jml+xuoXF2YTY8y6cqnJmf2muWe5noloZj6vPhj1d1ATzSTXtjciDDA9e8/CTWSxcs+okZLYYLZ9PqWLxRUOj5/MpS2cSNcyx01hXavpUwgZESoGi52EQDrLUWzLLGLpPIsoFerafFUGzohuYv45m4pvQqvXlH2ZQOIiihF/JdLneBqmcJclSoU+CVHkAL9vxAMvYhJmbEbMzNGUPc66IA/GUPfqlzj3jRglyfwxpqn1Jep6mNFFzPvwVg3qlSWSYcs5h0ZVbqb5zCyCmOV0XhZOy7LwfCyl0tnI4jQWW8WmoBXefgQI3dcmxatTV2k6u6bF+CyHNvhhzl5zluzBqWH8S7J/iQn3Zp50zb78XNCKCbodL0TBiCOpnksF5ReiSP/i7fs/10IPmiz5Khh+acIMQerIBUsoRllwsztm375lRuJXMZCF3XA3ce8zz5jPek0xWvUdrQw+p1vgjtCDEYHVXb+D4fd6HofqirwKONHtiHkta2ac+GXK2Ufsc22Fi3I/6Oq19dYN8nFzi6y2frWG9LUfbV67Sq7eICubrdW1bbLS3FxdIr+6vg250mZrpbX2a3hOk2xvNte3mivba9fWyUfNtSvXN1tkC17SWo+ntl08DcFitTxX9dQzUDtNrc1SDFMZnVAb/pwGLvfZAQuIQH8U00uPQ8IfrNP6XLBZ3KXGPnQMn9M3OxFzNLeD/swz3wHEdWkRB3lOBqlyzMxfktObgFu3Z3JqdkGt+9lbeXV/BpS6Wrk2qTwe7Yad2t5SEoQd+/su7JmWiKxzGnqvaCdsdtnR1AoFauHReiSVCHBhrI/eMzuNs2KdT81Va3Ls3BwOOWOnC4cdIfW9xsx9lZSqIsq+ihYm+2HqmhBhbF9qEQhHrsIfEFSQ1eaNLdK8eG2ZfHyptU4+XrtyJYhHMFrZbH10fX21kTz2yLncQFUmuvUTQ6zrM7B8gypF+wJlvM2CxVisgPOOpkvr1iVars6i1p2LRqdaAk2TB+xX7faIZlajIa1tM6XCoUlHWGB1TMeBdSlrmuhf0QGWHcPB/5YX/hwW6tX4LGwigYoZmT+msPgJwrMICCzkquJkkjiHacyMPoOD0ciLcJbLqKpRMlj8OxAZpEZvocvVMZXaLJbBhbaWZ/UxZNqhMAb4xzP209DSnDoITU19TitLRnNswNlctUPqjoqpSUjcVoyB0TEIN33XLyfsm+Z3LQoFybGumwdExHNYf8dYm+4e12yP8PsSxMCbjmWQPhKDcCgM2xgJy/G5jhZLFTnKAA+3Z3uS3lGwOFh4o+B7g9PN9SzarWBYTXiF2TesXhbONB/ItLKqd1yRATJ8hkoRlyNfDa4JrBMEsQPPJ60RosBM7gzzQ4m8JjFZzWmRXUoc6uram0wwv4ITTaiPacOZNJzG2fFkpXeBdblSdHXs16txA3RMbMbA70+LZ/0x6wTyI8iEyXyRXY7Ucz0cJ2Zh7WgwNKxONkcvz8JGRX1Sztj8lPa78PobZ+HG0qsrFSuVOcrTNxZ7cLJeTIS1a8yKhstYolvXo0yg01Ph0IFZ+k8UYPMFrYhxkvxDXxL9Hvvrc2ko6GU0o7kTcDecJ+aubFW2VDhlTdZLvWoPzDH3MjuW7GS4hCo2TJmPCalmRjkrwbwL8NovjiZakirNlYcmuzQ1Bzrp8rivIkxqLPqG0fVyHHhSNC9CtHff8+afaIYEk1lMkPK7KOAyZjgrMIURQBVDTwm9QsHVd470cXIg7qyW55AVjecBiskQ1J0lvYQSYKNXQ9QCkIZTiHybp9qNGUHrKUpGCOD6/wdAcaQQ+4kBAA=='
    EMBEDDED_VAL = 'H4sIAE27y2oC/81a3W7byBW+L9B3OHFvHZekJErKjaH1pmsjjm3kB3tpjKiRyIgiuUPSjnZRIJsif1gEaYF1dhukdQNIqeHdJIaBBDJQpK8i9AXaR+iZGZKibMpmgkTyjSjOkEPO4Tnf+c7Pd3NWY+7SXGAZbRpsKoqiVgpz83MBvR3gcJ25MOztDHv/HvYGw/4dPNkb9h/g4TUehr0DefJ02Hs/7PXSM3fx3z1+de+f4u6dYf9PeAmIe+4Oe/sgVuQ3HA77j4a9n8Rle/JxB+IGPvnLsP+DHMal8L5fcRLE4SXeCeKA973BN8CTo2H/4bDfg//t/vUJ7sImTiskLYo7MS2nYeGQQQLaclkXh5qMhI1NRj2XBTjjMctlVsBnDDxaBrFxlNFvQotRf9MMO8SZuxSwkM7P+W7IDL6s33UCk+LFeOkWsa0GLo/ybBLbx8sMRvn5JuGy1BRNv6gqFxX9hqpe0rRLanmhXKoWFW3uj7/9zXcnvoNWHn0Hk3YhoKQDGzYlPgVcuGEF0OkCo83QaUDgAr4H4AZalkNs8Ei3Qx28ggam2wDL8fH2BrhN8AOXUdgmtk2DBbhwYVxM1GnZlm+OC0o+Y5OLgvrHRNXBFwk7n19QRVUtlzMFpZULGYISevgw0cN9oT/vxTDX1YFUSuAz/cdCKe+IOalLr4V2opbxuYFQRnERn5Y3vJNKecAVkJ+/TBQ3es4z8ZwjbhaxaqL6fi+18+dXZ2qnH9Z91EMvsFwnp8gjYX465dR0LVvmVa0ykvkqLuMHqGNbrmVQVDkIHXrbowauvnhinxn6VbdsGydmo1ilUqVazdykguPJJgVAHUoE249V6K1QHPz6L3D0hRg9TBAyArBfxfDLFGQKdbgn1AVnHglY201wcEfqyg8gnpjCtXmpvC9TuiaAlmtUjK/7yR0Ixt+Lh92V7zwQ/56Lt/27ePCefGQaq/kk3t+PLYUv8Te5jUdi8VcCjaU1RHotAfpA2swI+99FqM+f9Ea+Qmwewo3sC4H9S1z/Eze+fu9MkyCG4YZOsIlH6vuzguxiQS2rmQpTUZWRwmwQo407AcKYtUU5QjMHiHPcIjIBt0FtvId1Ny3fD+n4Rm13ewqmrytFtZK5SV1TxzdJA/BMEhAwQ4JOiAAJGd8oYdAm4BN8P2iRLqnzyYU8eHDa9k2rZX7+b6xrRS37G2sFbbT9r3Hf+IGp9L0eZSTADYGJr+FzX9vFP2CE6HI7lIEfepxqgIGOmbI8etCiDq5pb6LfxfNZqIFeLmWrgaan6MkN3P7NjRUIGHF8YnCXhbtFK6UN1Hw8+cp1WzaFDdKdh3oYRIIxqdF28czjduKb7jZejMpCGXPZQg7xRCRns0ksO2QzUZSiourZAioratp77CXInO09BgIJd9Mw+WDxTET0iO9vu4xzM5/OipnparmULYJKaSSCFXAoh0FUla+u34CA3E4YQ9NlkspyywAP38vkJJcP10PfchDsoWlxirAAK2CSLQp1Sh0wXCfg2oYmd8LI+M38WTZBalKABun6sG0FJjguPsh37ZBraR4ty2Qn09GuUkEvFLKjAyVlfl9GeAl8fQRdG9oWbt4SpubFfqiOAwFpA7c4Lp0YZcEhOIPgnQubUdko20RDN9r5KdsnJ6aKrujZclEL47DUcfH74e49z0Z6IJCpTann54HfgBqmw0nF7BxRCW0oWwMKpUIaXw5FtIPYsitY5L6khJxpvRV4I5ngrhz6Ec/PBpdTvM/UvvRkB1RUiplh3wtBPA9FPCcR9o5g4Vwqb0b5C5lA6AnKekcQ4zsjkf0okww/y8O+PAyEHF8n3PmlQOxouUGc+4jDvx150yg7EQV+L84U+zlgP+VyeYLSFccYruUTRN4ACY9geLmCvY9x2p9aqyYnE6pVfbTBq+Etk0OHg5xWhh6AHA/fCN/f7nIApQJvGV5gEksSX9vmMEyQxYhEjYng2qULcAVD+S4By2+7ELJWvELbpJaDITMjuE7TAids+0iBUKCMmGLVXKgcR0aG7fonhJpNFT+1zpQwZsgUKTqxkUivo5N2HQoyhEM+gH4/evk8iHwuUnaVQiU7NlDSkFzb2IAvass3b8D11fWvYWm5tgrXass1WK6tQO3mNbi+dO3y5TX4YrW2dgWW1/nkCp9cWFxcvHDheGIu+7uf6qGmEyJMFodWHmPAY+mEJ3EWYRAT4jfZKPtKoOzz44ne8dQIcACXOeoYwnuR1+Pg/lRmj2U+4lDk5virRFdGOeOzuTZzG6GBNuZ2PJtYzozo9mRDU9MRxxXLR1LHw3OGxFsiDKMty8fQE82OdhB+BWDFIQSHKqcl4AwaVldiz2eGrfOdgS8U9UnODkWCmxcwhk4Bcd5DqQlqjAFIp0OQVXdwkKIS8XxATnGc6hqnmOCanGpXS8csepTQlMnxXwTNfCFoJ2dch6mwNk9A+1GlhunRA0VP5b2XUOO7bggIDFtWA0NNpG15nNi5CKGKVbVSzFb7sp6O2rcJanjgooaHPi8wgUEcAyMqdNwdjL9NxAOP0Q6+NaRLFnki6/NQ4lDLqjLhU5fTiT5u7ZYfJfpEGMllMsYGeSpii0fgxGnE5LDDpeM2MQRdnEHq4lRaODWHVdTUCQCrjxGEZwIr0EXfF+WGXRlWPR5V37gv3xPlZl5yOEwqzkfSkb8Ww7Le8kA6+j/LSFiOHVxM/kFqUQyb341qLgeiZvf+A3jB+c4SpPMhnv0tmNT24Cp33Injr5uua4sADsxwHkQqEcHMaUcswaPo4VClJTO4JcKZzBJmpks7BwnK8qSCjVJKKeDV2soa1DbWLkNtaWn95toNuHoZVte/wtE1Ts2v1K7BRk1w+PnkktX1pSuXv4Try0jxJYevfSiHP62qNaUkdlHVqxOSTGMl0Ig+Z6dYjmSnhjTJt1ExVMz05eEwtu+oz+OeYPcRWd+RJokrPJT/7saUIgoM9qSdy+ri/SSncqZ5nuZtp2SdilrJFm8p3eOxLugj967UjklmYJJ58GTPR0Thm2iJt4jdsMBykBHKjC9zcxHM09jVlFRNU/VsalVMO4MV3DQxRLkAna74hPmZlRThDHIgk4FGKxbTmyMdCB1SRxaFJMJ2W/gp+b9RLmQBVjCs4I4/Tjz5oc/DCV5OY1C3XVy5sTBDSpGFVtMjrxO1SC2lBF3nSbkxsxrl6joit9fuhk7bQt9n80Juo8F4qclH6mYkSU1Rtc5uKPsgDZyat5tI7DXlWMk2YQAjl7+Yq+w62adPJ+dULE0ytGq65JjwnYQIjvpTdmR3WNwJdihZ4m7SajjyNTLv9PQEN5xPSGSUibovn4EPGODhL0l1N+Gbkl+KDhqIclUpCstfZxH+8+wf/x08yenVsvRsSjheUcvZdRk1DXV/IB2LB4i4GdEX5rUYaciUOf/lsRGHEbRDYZW5CgiTo8ZZF2WVajHNljJavCIlOIo18V1cLIoaYbmmPTnGsMZ7AkZFqrhxcaw58U2y4OOxDOli7uaqGZYQigjs2T1HpXJW3ulk73FUhjsQY7wFTcpuV2aMH47SxVFN77iRogk/5CdCwM8FEBwkGawxVDgSKeR8iePz3KZSKSnHnIJMc8ftAUg4kIGJnDcNKNqy0BLKmtQKRKLDc5FbfBPiSwXdWRCSM9LyU6qYTkzKK3p1YhEk7kuOwqeofXRkxU/jpEfU8PkeZKFalE8GSTV/V/iOqJwiHdBenH+NnFHSjMzxBXV2Unh6XHU/rDP3kyfnirpWndDmoWa4+bjFyI3pnszI/65a0dQiEB8vkNwu6kPaZi7qZtIHE/G/nE3x54DpTQw31EKKBW3EBh3KFhj6+yZpU3Csti2c7nxsvVAnJq9vWGi+PNeTu2zxkYWx6fWKlHQt3Sti2y74FoMa8ZD8J9hkiE5O3kWCzAQJLs8XQGB1OFOp8yIPP/AOKs5UcmfAZt/DGXdz/B+4z/b66zMAAA=='
    EMBEDDED_TEST = 'H4sIAE27y2oC/9VaXW8UyRV9j5T/UOsk8otBM2MbbL8gDErkVWAt7CjJk1XTXZ6unZ6uTlW1h2EVCYwWFiHESwwRSmShjA3yemFkyStbiuCvjPJLcu+t7uke4w1tpODeF7u7vvtOnTrn3lvfTEh/YmHCCmPXarX6xBQ83rZQsqp5ZLhnpYrY0nW2+qebF+bn5+psnctQ+Ixb1uRRm7W4FV3eY83Eso6KRI+1lGW+aEor/IswXMijVsJbAoYUUSuUJoBCD3q1lO5BYcx7HRHZNRw30QIqYy2VlhYrA9nC5lr8JZFamLUg6fBoYsHqRExNGJVoD8f1Eg3j+dBwg4fSp+e0jacFvq5x/KRGrXHpQr12oXZptV5faDQW6vMXG/WZxqXaxF9/+YtvxizRyC1xQ2jOVhaXGPc8lUSWGcHqjVqNtcEILd4TLOaaKe0LzQxMKCIWCya0Vppxji34CUME8mdkiencEsP+3nDn4bD/hv5uDvvbbNjfGvYHw34fiuDlCCv7b+HpObXZHvYPXQ08vRv2qXP/wbD/yHU+Hu48psb9t1APT3+D0eDf/nDnO5phiwalEfaoLH3ZohFfj16w71Oa/ylNgc9Q9CAffLjT/+BX8OXP4ieYyX+CazCPhI0WS68NMExipoVNdATPJlDCsBnm855hvKUusuVQcNisMpJWwszQdD2JfGYVYbcMOl2PNfxmWMq4PTrCl0nnFIus89D8v00ym5tkMeAytQKZBWwCXxgoQh6zASd88mYgmeXtzAgZmDuCwSJ8aVnEoUWQlANrZQ1zqQjXQwLdnsMCoOQ9AMMh4qkD6zsCC/x9ncMowxm9fDvs/zDcucsQ4QRGh/2BA94z6ptiPz0VaKCdJwi6ncfu/TUB/a57eeDgeOWjYKysiS/nJv5DxJuhwP0WqhYAjfkJvU1m2ytUhNN14AIjYDGwfNiB3KjITLIu1xHsrjJATMdbg//CmKocTHO5Jb5aXWYrN1bg4zwhN0SKJsU0BwB2hBbsS6lYlHSaeHoBIMFgYK8213AwWdkRpWBXTTPMj1MkYuKIwDcgILx0WDt2OHpJ9Qcj8tpzjEcomyTAbVGLRyOGPSLWe5Kx7yaOgQ8vJ6Fyl1D9zKF1gLPQ0zGCuATjVdKg9Vpu0GVuTBfEFWwsIywTHaBmtiG0XJfwGahOQwk6tKlhvxjkQ85majNM3I5hzT7stJYoJ0PdNGs0TXUOm3pBkv9WaRTX2VLdh3shUB7CKBIkO8k+BD7QngQ+GTXVbQZUaGLeYSWBVgF7zDbqjQ/t0ShiDYjmPiEoVY9j5LRHUPkHEY9r2B81OciBdzpeB9TtyInYkcod15sHBL1UpAI27xHT3XdgfkhTE1xLCc+Kmrug/lM1yZtKW8ZBQnk88kSYej6/uny53pieZ7KD64XJwh7jhi25akCin8QhQrYUHKnTmpug1KH0OWxRkOF/FkH63S0eWokuYRxyT4xkJ2jSKeZLgJz1AigFdGIbEYBa0KoNVV/LXjkoVtIWs0UYvsm8tRQfzwkbfUdIUPw8x8wjkotPMtjtQtELh5wRoN7mDDZVEJNUCP+OnYZMoeow/Jo6DjKe7Oee5RHN3M+H/Lju/Gl7nysSC8J+NQDJpLnXhv3Crv5xkc3N1Rvztelp9AG7BhjCa8PnMWMT4AZQWRaDOeDiBEmTpChnXSHa5YGYTVYhcxRE+EogY3TbgQWBEzfwu7lNDPMCkVJjikcvdZ6bqodtPRVZ7tlMlJ4ZkhW0ytwJWBIbpl4d4s2h5sgR4EPnF+YxG+chYtmPJCQHY14doieP0EDb/dwr3KPqe4T2F9TjeSkfr7KGLEj6JTzH0c3zGU+suqBFJLo8ZD2Qo51eFj3wOCiyLjBeU4bY9NeNeQe1KEqgMbBDVAZvJmkaT8sYle3nV+On2qJRUOM34R/GeW2AACoulr4wkwSAubZgTW6k58pBmfqqG7U090WOyFJY+2mLnOcGadRPQxo5eHuEMScg32SM9sYBbTejqLcjUnLvh/R+33HZIfV9fUK55rFS4sU3hLd/OsZ9OAqMDnASHHG3oFrP5BJW1OAF4X9dYpw9hr3WQ3ZDxMHWWYDnDSU9inOGwrOG1ed+wyy/zWB7wr8L4rboxBjy014AAlaaUlo0Hb6kKT4DHguafBHWhiHM/2weN2ZryGgbIpICQAjsxzVogLYkIhR3iB15i4DHfGHBT8Qa8BRLkt6pdghV95z2w0wpR/Cui2FuErzeOxw+y9C5m3Hi3VyggrR8RS230kp6pgQGilynRo8c5BxUH42YdI9GfUl/nw7733+KJ1gxK8+Oq840QYInf0cag+e+iPxYSSizgUbxydmXK1/dBG9HS1jFHRejma3VXEasDOCs8IIIXMVwDaZIRFVosCDAr0a+VhL0QByzJMbhGfEdR6dYW4o6Ied1AwlVaVKQc2Q+0AvcBFlktBTwPsUen2NvXB5PNeAW38+yCmnYhGIsk1S4n7Hh9iQl+Ch/5wKY+y4FMRYP3aLswXcFPk0RvJW95q7iWH4hpcnBKOBalvf+p5nPlfoKqv7araXVpWtXf7/AbiShlXEoWBKBLA1goXdAdrpT3+ABDwyImQetOmxpmanEGgniCxUrRnx174sSUFzXPMGgVKz0iagUKASLxjofLM6fSP8By0kWCaR2PkrtBdy5f220UTsC/cm478NCDWLThW9c1AZx6cse/6IMHCtqk+laEY7fOrg4vbjp0gsAioWTQdIMTZRPcPBkOffl2Qtq/IDQuTVKxz/O2myPRVRfUY3LDb6iYf9Ns0OPzazHvyjcs+NIdNc9sMyJfHAG1Fb11yj4B9cFRiV0bxR/6OCe9ClMI/CyDFW7SzX1Olu+QbdpIgUuk9JNAQoP/UqdlMsUpsP1KkWe041TzIFhGMxaODMYAW6yVcpiEh4UqgRvG91nH7AZCO7HGEQ1jGRH1AK1UU61VtMcY5dqHO4OCCOHIz16mOYJCWw5N54M4jjuPM6uzeSa9cex4M33Wepwa5QzRPKcOj0kO0JqmvgfuLCrO0Sen+1OTTV/gJliDh+J4HfXl28xDeuxZootsfTWARCFhiWh5AVmMYkG3lhHDo21WpdAvpgHsSIUHQGMepZEvhcq88EVo3P0KqcLSv9G8jVofR5HOZfCryhsIXRD4DScg0EM36DDzHkGYIhA+XBciY7aEGliEjpEZ8rvn2qdc/OCpi+dku1wjuWei9MUwz6PEW2pBj7IA6ZHBL99+us8zL4r/SG7F5BemHuXqt9RbGiHRnvrmg0Imns5KR9nd+vSMM+L7EDIkx7Q4d5ZLgNUzPoFP+OWu13iMxWDR4V57a60AXAHEig6WphuE4BgLxRchz267aWbsM+gj4yMBSJBBEeiW+pWgFZ+4oFBVAeEIji5VTm+Ch7BCmAQccbWeVNLD6Nh7SR0KZ8Y6gCTiFWrdMQi2Q75FOuKpkHHFCjVKsxL8pC3SgdkP2KU8/SUpsdu4mSxF4JBfsX05LW3p6R3d11s6O+Epvcn76NuE4i2R4gqiNQpokgsccnOvuPh3Uzflsv/f9JG+wwWnSl4FdcV66kE4LMOfAn7SmhYIuyexcYi21Bh0hFIi7AIQUzgS0PHiXG337xAqdBcKYG6loiAbMM1YGB4r8YhNFPQ8ytJjN4G88gGLFA4R498SjyVtAis48h2jzNwQDCeh/GhUPbElTIIq6QBTl68yZPyI7fwLsuuzByO7qMdEpqcmB3kzJaS3iPqekAXvmHAV6M7bP08dvPxLGJF7PVfRQmiQ0wxAAA='
    train_file.write_bytes(gzip.decompress(base64.b64decode(EMBEDDED_TRAIN)))
    val_file.write_bytes(gzip.decompress(base64.b64decode(EMBEDDED_VAL)))
    test_file.write_bytes(gzip.decompress(base64.b64decode(EMBEDDED_TEST)))
    print('Self-extraction complete!')

# 2. Verify Cryptographic SHA-256 Hashes
EXPECTED_TRAIN_HASH = '11df262072d7243f623bfd829b407a8dbd2c5a3087ed697a836a7152c7688967'
EXPECTED_VAL_HASH = '2e349ad3b1330aec224c266460d2e4189b94138a1d63146a10bd8f45ea65508e'
EXPECTED_TEST_HASH = 'edf3fa5cd949a149ad5d759d81553d9704467491877d0f3c8b7cc5a887a9eda4'

act_train = compute_sha256(train_file)
act_val = compute_sha256(val_file)
act_test = compute_sha256(test_file)

print(f'Train hash:      {act_train} -> {"VERIFIED" if act_train == EXPECTED_TRAIN_HASH else "MISMATCH"}')
print(f'Validation hash: {act_val} -> {"VERIFIED" if act_val == EXPECTED_VAL_HASH else "MISMATCH"}')
print(f'Test hash:       {act_test} -> {"VERIFIED" if act_test == EXPECTED_TEST_HASH else "MISMATCH"}')
assert act_train == EXPECTED_TRAIN_HASH, 'Train dataset hash mismatch!'
assert act_val == EXPECTED_VAL_HASH, 'Validation dataset hash mismatch!'
assert act_test == EXPECTED_TEST_HASH, 'Test dataset hash mismatch!'


## 4. Tokenizer & Model Configuration Inspection
Verifies `model.config.vocab_size` (151,936) vs `len(tokenizer)` (151,665), and ChatML templates.

In [ ]:
from transformers import AutoTokenizer, AutoConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading model config and tokenizer for: {MODEL_ID}")

config = AutoConfig.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"model.config.vocab_size: {config.vocab_size:,}")
print(f"len(tokenizer):          {len(tokenizer):,}")
print(f"pad_token:               {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"eos_token:               {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# Sample ChatML formatting
sample_msgs = [
    {"role": "system", "content": "You are NEXUS-SLM."},
    {"role": "user", "content": "Payment debited but order failed."}
]
formatted = tokenizer.apply_chat_template(sample_msgs, tokenize=False, add_generation_prompt=True)
print("\nChatML Formatting Sample:")
print(formatted)

## 5. Dataset Tokenization with Completion-Only Loss Masking
Crucial standard: all prompt tokens receive label `-100`. Only the structured JSON assistant tokens and the terminating `<|im_end|>` token receive valid label IDs.

In [ ]:
from datasets import Dataset

CATEGORIES = [
    "payment_failure", "refund_request", "account_access", "password_reset",
    "order_cancel", "order_tracking", "subscription", "billing",
    "technical_issue", "fraud_report", "delivery_issue", "account_closure",
    "product_complaint", "general_query"
]
PRIORITIES = ["low", "medium", "high", "critical"]
LANGUAGES = ["english", "hindi", "hinglish"]
SYSTEM_PROMPT = "You are NEXUS-SLM, an enterprise customer support intelligence triage agent. Analyze the customer support ticket and classify it into the predefined taxonomy. Return valid JSON only."

def format_prompt(text: str) -> str:
    user_content = (
        f"Support Ticket:\n\"{text}\"\n\n"
        f"Allowed categories: {', '.join(CATEGORIES)}\n"
        f"Allowed priorities: {', '.join(PRIORITIES)}\n"
        f"Allowed languages: {', '.join(LANGUAGES)}\n"
        "Return valid JSON only."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def prepare_masked_dataset(jsonl_path: Path, max_length: int = 384) -> Dataset:
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    input_ids_list = []
    labels_list = []

    for r in records:
        target = {
            "category": r["category"],
            "priority": r["priority"],
            "language": r["language"],
            "requires_human": r["requires_human"],
        }
        prompt_str = format_prompt(r["text"])
        completion_str = f"{json.dumps(target, ensure_ascii=False)}<|im_end|>"
        full_str = f"{prompt_str}{completion_str}"

        prompt_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        full_ids = tokenizer.encode(full_str, add_special_tokens=False)

        prompt_len = len(prompt_ids) if full_ids[:len(prompt_ids)] == prompt_ids else max(0, len(full_ids) - len(tokenizer.encode(completion_str, add_special_tokens=False)))
        labels = [-100] * prompt_len + full_ids[prompt_len:]

        input_ids_list.append(full_ids[:max_length])
        labels_list.append(labels[:max_length])

    return Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})

train_dataset = prepare_masked_dataset(train_file)
val_dataset = prepare_masked_dataset(val_file)
print(f"Prepared Train Dataset: {len(train_dataset)} examples")
print(f"Prepared Val Dataset:   {len(val_dataset)} examples")

# Verify mask on sample 0
sample_lbls = train_dataset[0]["labels"]
masked_cnt = sum(1 for x in sample_lbls if x == -100)
active_cnt = sum(1 for x in sample_lbls if x != -100)
print(f"Sample 0 Token Count: {len(sample_lbls)} | Prompt Masked (-100): {masked_cnt} | Completion Active: {active_cnt}")

## 6. Forward & Backward Smoke Test
Runs a single step forward/backward pass on 2 instances to verify CUDA memory, tensor shapes, and loss computation.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

print("Executing GPU smoke test...")
smoke_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True,
)
smoke_lora = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
smoke_peft = get_peft_model(smoke_model, smoke_lora)

dummy_batch = torch.tensor([train_dataset[0]["input_ids"]], device="cuda:0")
dummy_labels = torch.tensor([train_dataset[0]["labels"]], device="cuda:0")

out = smoke_peft(input_ids=dummy_batch, labels=dummy_labels)
loss = out.loss
loss.backward()
print(f"Smoke test PASSED! Forward loss: {loss.item():.4f}, Gradients computed successfully.")

del smoke_peft, smoke_model, dummy_batch, dummy_labels, out, loss
torch.cuda.empty_cache()


## 7. Full Supervised LoRA Training
Trains candidate `qwen25_05b_lora_seed42` with $r=16, \alpha=32$ across 7 linear projection target modules.

In [ ]:
import time
from transformers import TrainingArguments, Trainer

torch.manual_seed(42)

torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Loading base model in {torch_dtype}...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
if hasattr(base_model, 'enable_input_require_grads'):
    base_model.enable_input_require_grads()
model = get_peft_model(base_model, lora_config)
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.print_trainable_parameters()

CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    gradient_checkpointing=False,
    seed=42,
    report_to=["none"],
)

class DynamicDataCollator:
    def __init__(self, pad_id, max_len=384):
        self.pad_id = pad_id
        self.max_len = max_len
    def __call__(self, batch):
        max_b = min(max(len(x["input_ids"]) for x in batch), self.max_len)
        inp, att, lbl = [], [], []
        for x in batch:
            ids = x["input_ids"][:max_b]
            lb = x["labels"][:max_b]
            pad_l = max_b - len(ids)
            inp.append(ids + [self.pad_id] * pad_l)
            att.append([1] * len(ids) + [0] * pad_l)
            lbl.append(lb + [-100] * pad_l)
        return {
            "input_ids": torch.tensor(inp, dtype=torch.long),
            "attention_mask": torch.tensor(att, dtype=torch.long),
            "labels": torch.tensor(lbl, dtype=torch.long),
        }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DynamicDataCollator(tokenizer.pad_token_id),
)

print("Starting training execution...")
t0 = time.time()
train_result = trainer.train()
duration = round(time.time() - t0, 2)

eval_res = trainer.evaluate()
print(f"Training finished in {duration}s! Train Loss: {train_result.training_loss:.4f}, Eval Loss: {eval_res['eval_loss']:.4f}")


## 8. Export Adapter Artifacts & Hashes
Saves the fine-tuned adapter, tokenizer, configuration, and SHA-256 hashes.

In [ ]:
EXPORT_DIR = WORK_DIR / "nexus_qwen25_05b_lora_seed42"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving trained adapter to {EXPORT_DIR}...")
model.save_pretrained(str(EXPORT_DIR))
tokenizer.save_pretrained(str(EXPORT_DIR))

artifact_hashes = {}
for f in EXPORT_DIR.glob("*"):
    if f.is_file():
        artifact_hashes[f.name] = compute_sha256(f)

metadata = {
    "model_version": "qwen25_05b_lora_seed42",
    "base_model": MODEL_ID,
    "seed": 42,
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05},
    "train_loss": float(train_result.training_loss),
    "eval_loss": float(eval_res["eval_loss"]),
    "training_duration_sec": duration,
    "artifact_hashes": artifact_hashes,
}

with open(EXPORT_DIR / "nexus_qwen_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Export complete!")
for k, v in artifact_hashes.items():
    print(f"  {k:<30}: {v}")

## 9. Validation Evaluation & Candidate Acceptance Gate Verification
Evaluates the candidate on `data/validation/validation.jsonl` ($N=44$) against all 6 candidate acceptance gates.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Evaluating on Validation Split (N=44) for Model Selection...")
model.eval()

with open(val_file, "r", encoding="utf-8") as f:
    val_records = [json.loads(l) for l in f if l.strip()]

predictions = []
raw_texts = []

for r in val_records:
    p_text = format_prompt(r["text"])
    inputs = tokenizer(p_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=60, do_sample=False, temperature=None)
    gen_text = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    raw_texts.append(gen_text)
    try:
        parsed = json.loads(gen_text.strip())
        cat = parsed.get("category", "unknown")
    except Exception:
        cat = "parse_error"
    predictions.append(cat)

y_true = [r["category"] for r in val_records]
acc = accuracy_score(y_true, predictions)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, predictions, labels=CATEGORIES, average="macro", zero_division=0)

print("=" * 70)
print(f"Validation Category Accuracy: {acc:.4f}")
print(f"Validation Macro-F1:          {macro_f1:.4f}")
print("=" * 70)

# Check Gate 1
print("\nCandidate Acceptance Gates Check:")
print(f"  Gate 1 (Val Macro-F1 >= 0.55): {'PASS' if macro_f1 >= 0.55 else 'FAIL/HOLD'} (Measured: {macro_f1:.4f})")

## 10. Packaging & Retrieval Instructions
Compresses exported artifacts into a zip archive and outputs download commands.

In [ ]:
import shutil

zip_path = WORK_DIR / "nexus_qwen25_05b_lora_seed42.zip"
shutil.make_archive(str(WORK_DIR / "nexus_qwen25_05b_lora_seed42"), 'zip', str(EXPORT_DIR))

print("=" * 70)
print("ARTIFACT RETRIEVAL COMMANDS")
print("=" * 70)
print(f"Zip archive created at: {zip_path}")
print("\n1. To retrieve via Kaggle API:")
print("   kaggle kernels output <your-username>/<kernel-slug> -p models/qwen25_05b_lora_seed42/")
print("\n2. To unpack locally into NEXUS-SLM:")
print("   Expand-Archive -Path models/qwen25_05b_lora_seed42/nexus_qwen25_05b_lora_seed42.zip -DestinationPath models/qwen25_05b_lora_seed42/")
print("=" * 70)